In [ ]:
import os
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

DATA_DIR = "data"
BATCH_SIZE = 32
IMAGE_SIZE = 224

# Resize and normalize only. No random crop, flip, or other augmentation.
transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)


class NumericImageFolder(datasets.ImageFolder):
    """ImageFolder that uses the numeric folder name as the class index."""

    def find_classes(self, directory):
        classes = [
            entry.name
            for entry in os.scandir(directory)
            if entry.is_dir() and entry.name.isdigit()
        ]
        classes.sort(key=int)
        class_to_idx = {name: int(name) for name in classes}
        return classes, class_to_idx


def painter_names(root):
    names = {}
    for class_dir in Path(root).iterdir():
        if not class_dir.is_dir() or not class_dir.name.isdigit():
            continue
        sample = next(
            path
            for path in class_dir.iterdir()
            if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
        )
        names[int(class_dir.name)] = sample.stem.rsplit("_", 1)[0]
    return dict(sorted(names.items()))


dataset = NumericImageFolder(DATA_DIR, transform=transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
num_classes = len(dataset.classes)
painters = painter_names(DATA_DIR)


def replace_classifier(model, num_classes):
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


# torchvision has ResNet-18, ResNet-34, and ResNet-152 (not 8 / 150).
resnet18 = replace_classifier(
    models.resnet18(weights=models.ResNet18_Weights.DEFAULT), num_classes
)
resnet34 = replace_classifier(
    models.resnet34(weights=models.ResNet34_Weights.DEFAULT), num_classes
)
resnet152 = replace_classifier(
    models.resnet152(weights=models.ResNet152_Weights.DEFAULT), num_classes
)

print(f"samples: {len(dataset)}, classes: {num_classes}, batches: {len(loader)}")
print(painters)
print(resnet18.fc)
print(resnet34.fc)
print(resnet152.fc)

In [ ]:

def replace_classifier(model, num_classes):
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model



In [ ]:
import copy
import time

import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split

MAX_EPOCHS = 50
INITIAL_LR = 1e-4

val_size = int(0.2 * len(dataset))
train_size = len(dataset) - val_size
train_set, val_set = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
print(f"train: {len(train_set)}   validation: {len(val_set)}")

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("device:", device)


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        if training:
            optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = torch.nn.functional.cross_entropy(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, correct / total


def train_one_model(model, name):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=INITIAL_LR)
    best_acc = -1.0
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        started = time.time()
        train_loss, train_acc = run_epoch(model, train_loader, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader)
        lr = optimizer.param_groups[0]["lr"]
        print(
            f"{name}  epoch {epoch:02d}/{MAX_EPOCHS}  lr {lr:.1e}  "
            f"train acc {train_acc:.3f}  val acc {val_acc:.3f}  "
            f"({time.time() - started:.0f}s)",
            flush=True,
        )

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement % 10 == 0:
                optimizer.param_groups[0]["lr"] *= 0.1
                print(
                    f"{name}  no val improvement for {epochs_without_improvement} epochs, "
                    f"lr -> {optimizer.param_groups[0]['lr']:.1e}",
                    flush=True,
                )
            if epochs_without_improvement >= 20:
                print(f"{name}  early stopping at epoch {epoch}", flush=True)
                break

    model.load_state_dict(best_state)
    path = f"{name}.pth"
    torch.save(model.state_dict(), path)
    print(f"{name}  saved {path}  best val acc {best_acc:.3f}", flush=True)
    model.cpu()
    if device.type == "mps":
        torch.mps.empty_cache()
    return best_acc


def per_class_accuracy(model):
    model.to(device)
    model.eval()
    correct = torch.zeros(num_classes)
    total = torch.zeros(num_classes)
    with torch.no_grad():
        for images, labels in val_loader:
            preds = model(images.to(device)).argmax(dim=1).cpu()
            for class_id in range(num_classes):
                chosen = labels == class_id
                total[class_id] += chosen.sum()
                correct[class_id] += (preds[chosen] == class_id).sum()
    accuracy = torch.zeros(num_classes)
    counted = total > 0
    accuracy[counted] = correct[counted] / total[counted]
    model.cpu()
    return accuracy.tolist()


class_accuracy = {}
for model_name, model in (
    ("resnet18", resnet18),
    ("resnet34", resnet34),
    ("resnet152", resnet152),
):
    train_one_model(model, model_name)
    class_accuracy[model_name] = per_class_accuracy(model)

painter_labels = [painters[i].replace("_", " ") for i in range(num_classes)]
fig, axes = plt.subplots(1, 3, figsize=(18, 8), sharey=True)
for ax, (model_name, accuracies) in zip(axes, class_accuracy.items()):
    ax.barh(range(num_classes), accuracies, color="steelblue")
    ax.set_xlim(0, 1)
    ax.set_xlabel("accuracy")
    ax.set_title(model_name)
axes[0].set_yticks(range(num_classes))
axes[0].set_yticklabels(painter_labels)
fig.tight_layout()
fig.savefig("per_class_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
resnet152_retrained = replace_classifier(
    models.resnet152(weights=None),
    num_classes,
)
resnet152_retrained.load_state_dict(
    torch.load("resnet152.pth", map_location="cpu", weights_only=True)
)

for param in resnet152_retrained.parameters():
    param.requires_grad = False
for param in resnet152_retrained.layer4.parameters():
    param.requires_grad = True
for param in resnet152_retrained.fc.parameters():
    param.requires_grad = True

train_one_model(resnet152_retrained, "resnet152_retrained")
retrained_class_accuracy = per_class_accuracy(resnet152_retrained)

painter_labels = [painters[i].replace("_", " ") for i in range(num_classes)]
fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(range(num_classes), retrained_class_accuracy, color="steelblue")
ax.set_yticks(range(num_classes))
ax.set_yticklabels(painter_labels)
ax.set_xlim(0, 1)
ax.set_xlabel("accuracy")
ax.set_title("resnet152 retrained last layer + classifier")
fig.tight_layout()
fig.savefig("per_class_accuracy_resnet152_retrained.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
resnet152_retrained_v2 = replace_classifier(
    models.resnet152(weights=None),
    num_classes,
)
resnet152_retrained_v2.load_state_dict(
    torch.load("models/resnet152_retrained.pth", map_location="cpu", weights_only=True)
)

for param in resnet152_retrained_v2.parameters():
    param.requires_grad = False
for param in resnet152_retrained_v2.layer4.parameters():
    param.requires_grad = True
for param in resnet152_retrained_v2.layer3.parameters():
    param.requires_grad = True
for param in resnet152_retrained_v2.fc.parameters():
    param.requires_grad = True

train_one_model(resnet152_retrained_v2, "resnet152_retrained_v2")
retrained_class_accuracy = per_class_accuracy(resnet152_retrained_v2)

painter_labels = [painters[i].replace("_", " ") for i in range(num_classes)]
fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(range(num_classes), retrained_class_accuracy, color="steelblue")
ax.set_yticks(range(num_classes))
ax.set_yticklabels(painter_labels)
ax.set_xlim(0, 1)
ax.set_xlabel("accuracy")
ax.set_title("resnet152 retrained last layer + classifier")
fig.tight_layout()
fig.savefig("per_class_accuracy_resnet152_retrained.png", dpi=150, bbox_inches="tight")
plt.show()